# Run EzTaoX on DP1 DIA forced source catalog

This notebook demonstrates how to load DP1 data using LSDB and run EzTaoX fitting on DP1 light curves. 

In [1]:
from upath import UPath

import numpy as np
import arviz as az

import os
import lsdb
import numpyro
import numpyro.distributions as dist
import jax.numpy as jnp
import jax
import pyarrow as pa

from eztaox.kernels.quasisep import Exp
from eztaox.models import MultiVarModel
from eztaox.fitter import random_search
from nested_pandas import NestedDtype
from numpyro.infer import MCMC, NUTS, init_to_median
from numpyro.diagnostics import summary

First, let's load the DP1 data in HATS.

- We only want diaObjectForcedSource columns
- We only want long lightcurves (length >= 500)

In [2]:
base_path = UPath("/rubin/lsdb_data")
cols = ["diaObjectId"] + [f"diaObjectForcedSource.{col}" for col in ["midpointMjdTai","band","psfMag","psfMagErr"]]
dia_object_cat = lsdb.open_catalog(base_path / "dia_object_collection", columns=cols, filters=[["nDiaSources", ">", 500]])
print("Number of DIA objects in DP1:", f"{len(dia_object_cat):,}")
dia_object_cat

Number of DIA objects in DP1: 1,089,818


,diaObjectId,ra,dec,diaObjectForcedSource
npartitions=208,,,,
"Order: 6, Pixel: 130",int64[pyarrow],double[pyarrow],double[pyarrow],"nested<midpointMjdTai: [double], band: [string..."
"Order: 6, Pixel: 136",...,...,...,...
...,...,...,...,...
"Order: 11, Pixel: 36833621",...,...,...,...
"Order: 7, Pixel: 143884",...,...,...,...


We will select only objects that have sources for `r` and `i`:

In [3]:
dia_object_cat = dia_object_cat.query("diaObjectForcedSource.band in ['r','i']")
dia_object_cat = dia_object_cat.map_partitions(lambda df: df.dropna(subset=["diaObjectForcedSource"]))

We want to clean up the light curves as well:

- remove 10-sigma outliers in each night
- bin the data into nightly epochs: take mean(mag) in each night
- adjust the magnitude uncertainties: add in quaduature mean(magerr) and std(mag) in each night

In [4]:
def prepare_lightcurves(time, band, mag, magerr):
    times, mags, magerrs = {}, {}, {}

    for b in np.unique(band):
        mask = band == b
        time_band = time[mask]
        mag_band = mag[mask]
        magerr_band = magerr[mask]
        
        # Remove 10 sigma outliers in each band
        sigma = np.abs(mag_band - np.nanmean(mag_band)) / np.nanstd(mag_band)
        sigma_mask = sigma < 5
        time_band = time_band[sigma_mask]
        mag_band = mag_band[sigma_mask]
        magerr_band = magerr_band[sigma_mask]

        # Get mean mag and magerr for each unique epoch
        times[b] = np.unique(np.round(time_band))
        mags[b] = []
        magerrs[b] = []
        
        for epoch in times[b]:
            epoch_mask = np.round(time_band) == epoch
            tmp_mag, tmp_err = combine_mag(mag_band[epoch_mask], magerr_band[epoch_mask])
            mags[b].append(tmp_mag)
            magerrs[b].append(tmp_err)

    # Format light curves for fitting
    X, y, yerr = format_lightcurves(times, mags, magerrs)
    return X, y, yerr

def combine_mag(mag, mag_err):
    ## add in quaduature mean(magerr) and std(mag) in each epoch
    ## e.g., https://arxiv.org/abs/2411.06617
    mag = np.asarray(mag)
    mag_err = np.asarray(mag_err)
    mag_w = np.nanmean(mag)
    mag_w_err = np.sqrt(np.nanmean(mag_err) ** 2 + np.nanstd(mag) ** 2)
    return mag_w, mag_w_err

def format_lightcurves(times, mags, magerrs):
    bands = times.keys()
    inds = jnp.argsort(jnp.concatenate([times[b] for b in bands]))
    X = (
        jnp.concatenate([times[b] for b in bands])[inds],
        jnp.concatenate(
            [i * jnp.ones_like(times[b], dtype=int) for i, b in enumerate(bands)]
        )[inds],
    )
    for b in bands:
        mags[b] = jnp.array(mags[b])
        mags[b] -= jnp.median(mags[b])
    y = jnp.concatenate([jnp.array(mags[b]) for b in bands])[inds]
    yerr = jnp.concatenate([jnp.array(magerrs[b]) for b in bands])[inds] 
    return X, y, yerr

Let's define a init sampler:

In [5]:
def init_sampler(fixed_params=None):
    if fixed_params is None:
        fixed_params = {}
        
    if "log_drw_scale" in fixed_params:
        log_drw_scale = fixed_params["log_drw_scale"]
    else:
        log_drw_scale = numpyro.sample(
            "log_drw_scale", dist.Uniform(jnp.log(10), jnp.log(100))
        )
    if "log_drw_sigma" in fixed_params:
        log_drw_sigma = fixed_params["log_drw_sigma"]
    else:
        log_drw_sigma = numpyro.sample(
            "log_drw_sigma", dist.Uniform(jnp.log(1e-2), jnp.log(10))
        )
    log_kernel_param = jnp.stack([log_drw_scale, log_drw_sigma])
    numpyro.deterministic("log_kernel_param", log_kernel_param)

    # parameters to relate the amplitudes in each band
    log_amp_scale = numpyro.sample("log_amp_scale", dist.Uniform(-2, 2))

    mean = numpyro.sample(
        "mean",
        dist.Uniform(low=jnp.asarray([-0.1, -0.1]), high=jnp.asarray([0.1, 0.1])),
    )

    # interband lags
    lag = numpyro.sample("lag", dist.Uniform(-10, 10))

    sample_params = {
        "log_kernel_param": log_kernel_param,
        "log_amp_scale": log_amp_scale,
        "mean": mean,
        "lag": lag,
    }
    return sample_params

And instantiate a multivariate model to use for fitting:

In [6]:
def multivar_model(X, y, yerr, has_lag=True, zero_mean=True):
    nBand = len(np.unique(X[1]))
    k = Exp(scale=100.0, sigma=2.0)
    return MultiVarModel(X, y, yerr, k, nBand, has_lag=has_lag, zero_mean=zero_mean)

In [7]:
def run_mle(model, init_sampler):
    fit_key = jax.random.PRNGKey(1)
    nSample = 1_000
    nBest = 5  # it seems like this number needs to be high
    bestP, ll = random_search(model, init_sampler, fit_key, nSample, nBest)
    return bestP, ll

def run_MCMC(
    objid, 
    X, 
    y, 
    yerr, 
    numpyro_model, 
    has_lag=True, 
    zero_mean=True, 
    fixed_params=None, 
    save_chains=False,
):
    nuts_kernel = NUTS(
        numpyro_model,
        dense_mass=True,
        target_accept_prob=0.9,
        init_strategy=init_to_median,
    )
    mcmc = MCMC(
        nuts_kernel,
        num_warmup=500,
        num_samples=1000,
        num_chains=1,
        # progress_bar=False,
    )
    mcmc_seed = 0
    mcmc.run(
        jax.random.PRNGKey(mcmc_seed),
        X,
        yerr,
        y=y,
        has_lag=has_lag,
        zero_mean=zero_mean,
        fixed_params=fixed_params,
    )
    if save_chains:
        idata = az.from_numpyro(mcmc)
        idata.to_netcdf(f"mcmc_chains/{objid}.nc")
    return summary(mcmc.get_samples(group_by_chain=True))

def numpyro_model(X, yerr, y=None, has_lag=True, zero_mean=True, fixed_params={}):
    sample_params = init_sampler(fixed_params={})
    nBand = len(np.unique(X[1]))
    k = Exp(scale=100.0, sigma=1.0)  # init params for k are not used
    m = MultiVarModel(X, y, yerr, k, nBand, has_lag=has_lag, zero_mean=zero_mean)
    m.sample(sample_params)

Let's finally put these steps all together:

In [8]:
def run_eztaox(objid, times, band, mag, magerr, *, save_chains=False):
    X, y, yerr = prepare_lightcurves(times, band, mag, magerr)
    #model = multivar_model(X, y, yerr, has_lag=True, zero_mean=True)
    #bestP, ll = run_mle(X, y, yerr, model, init_sampler)
    mcmc_summary = run_MCMC(
        objid,
        X,
        y,
        yerr,
        numpyro_model,
        has_lag=True,
        zero_mean=True,
        fixed_params={"log_drw_scale": jnp.log(100.0)},
        save_chains=save_chains,
    )
    return make_nested(mcmc_summary)

def make_nested(summary):
    nested_dict = {}
    for key in summary:
        for key2 in summary[key]:
            nested_dict[f"{key}.{key2}"] = [summary[key][key2]]
    return nested_dict

And run it all with LSDB, parallelizing the computation:

In [9]:
# Metadata for the MCMC result
nested_subcols = ["mean","std","median","5.0%","95.0%","n_eff","r_hat"]
nested_cols = ["lag", "log_amp_scale", "log_drw_scale", "log_drw_sigma", "log_kernel_param", "mean"]
meta = {col: NestedDtype({sc: pa.float64() for sc in nested_subcols}) for col in nested_cols}

In [10]:
result = dia_object_cat.map_rows(
    run_eztaox,
    # We only need the forced sources time band and mag info
    columns=["diaObjectId"] + [f"diaObjectForcedSource.{col}" for col in ["midpointMjdTai", "band", "psfMag", "psfMagErr"]],
    row_container="args",
    append_columns=True,
    save_chains=True, # By default, it is False
    meta=meta,
)
result

,diaObjectId,ra,dec,diaObjectForcedSource,lag,log_amp_scale,log_drw_scale,log_drw_sigma,log_kernel_param,mean
npartitions=208,,,,,,,,,,
"Order: 6, Pixel: 130",int64[pyarrow],double[pyarrow],double[pyarrow],"nested<midpointMjdTai: [double], band: [string...","nested<mean: [double], std: [double], median: ...","nested<mean: [double], std: [double], median: ...","nested<mean: [double], std: [double], median: ...","nested<mean: [double], std: [double], median: ...","nested<mean: [double], std: [double], median: ...","nested<mean: [double], std: [double], median: ..."
"Order: 6, Pixel: 136",...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
"Order: 11, Pixel: 36833621",...,...,...,...,...,...,...,...,...,...
"Order: 7, Pixel: 143884",...,...,...,...,...,...,...,...,...,...


In [14]:
# Create a directory to store the mcmc chains
os.makedirs("mcmc_chains", exist_ok=True)

In [15]:
from dask.distributed import Client
with Client(n_workers=4):
    df = result.compute()
df

sample:  57%|█████▋    | 861/1500 [00:05<00:01, 519.78it/s, 15 steps of size 3.45e-01. acc. prob=0.95]

sample:  74%|███████▍  | 1117/1500 [00:05<00:00, 749.86it/s, 15 steps of size 3.45e-01. acc. prob=0.95]

sample: 100%|██████████| 1500/1500 [00:05<00:00, 260.51it/s, 15 steps of size 3.45e-01. acc. prob=0.95]


sample: 100%|██████████| 1500/1500 [00:06<00:00, 237.31it/s, 95 steps of size 3.60e-02. acc. prob=0.93] 


diaObjectId         ra        dec  \
_healpix_29                                                     
2528665081895373202  609788873886662663  53.122135 -28.344551   
2528689212471481767  609789629800906756  53.041797 -28.283192   
...                                 ...        ...        ...   
2528746659668617474  611255141361778689   52.95461 -27.948373   
2528749594173367688  611255072642301964  53.137026 -27.863437   

                                                 diaObjectForcedSource  \
_healpix_29                                                              
2528665081895373202  [{midpointMjdTai: 60623.259895, band: 'r', psf...   
2528689212471481767  [{midpointMjdTai: 60623.258521, band: 'i', psf...   
...                                                                ...   
2528746659668617474  [{midpointMjdTai: 60623.258521, band: 'i', psf...   
2528749594173367688  [{midpointMjdTai: 60623.258521, band: 'i', psf...   

                                                                   lag  \
_healpix_29                                                              
2528665081895373202  [{mean: 0.391716, std: 5.687097, median: 0.627...   
2528689212471481767  [{mean: 0.358637, std: 5.232503, median: 0.517...   
...                                                                ...   
2528746659668617474  [{mean: 0.362393, std: 5.530225, median: 0.608...   
2528749594173367688  [{mean: 0.436642, std: 5.81116, median: 0.7276...   

                                                         log_amp_scale  \
_healpix_29                                                              
2528665081895373202  [{mean: -1.332533, std: 0.521583, median: -1.4...   
2528689212471481767  [{mean: -0.896507, std: 0.646982, median: -0.9...   
...                                                                ...   
2528746659668617474  [{mean: -0.761793, std: 0.487889, median: -0.7...   
2528749594173367688  [{mean: -1.015669, std: 0.670553, median: -1.1...   

                                                         log_drw_scale  \
_healpix_29                                                              
2528665081895373202  [{mean: 3.876166, std: 0.569365, median: 4.008...   
2528689212471481767  [{mean: 3.852157, std: 0.575212, median: 3.963...   
...                                                                ...   
2528746659668617474  [{mean: 3.696983, std: 0.609029, median: 3.787...   
2528749594173367688  [{mean: 3.912006, std: 0.552963, median: 4.054...   

                                                         log_drw_sigma  \
_healpix_29                                                              
2528665081895373202  [{mean: -4.340102, std: 0.236481, median: -4.3...   
2528689212471481767  [{mean: -4.34179, std: 0.245402, median: -4.41...   
...                                                                ...   
2528746659668617474  [{mean: -4.276573, std: 0.267832, median: -4.3...   
2528749594173367688  [{mean: -4.364739, std: 0.217392, median: -4.4...   

                                                      log_kernel_param  \
_healpix_29                                                              
2528665081895373202  [{mean: array([ 3.87616632, -4.34010161]), std...   
2528689212471481767  [{mean: array([ 3.85215706, -4.34179023]), std...   
...                                                                ...   
2528746659668617474  [{mean: array([ 3.69698263, -4.27657317]), std...   
2528749594173367688  [{mean: array([ 3.91200604, -4.36473943]), std...   

                                                                  mean  
_healpix_29                                                             
2528665081895373202  [{mean: array([ 0.00397728, -0.00065217]), std...  
2528689212471481767  [{mean: array([ 0.00072372, -0.00152951]), std...  
...                                                                ...  
2528746659668617474  [{mean: array([ 0.00089366, -0.00218195]), std...  
2528749594173367688 

We can load one of the chains as follows:

In [16]:
az.from_netcdf("mcmc_chains/611254385447534607.nc")

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:                 (chain: 1, draw: 1000, log_kernel_param_dim_0: 2,
│                                    mean_dim_0: 2)
│       Coordinates:
│         * chain                   (chain) int64 8B 0
│         * draw                    (draw) int64 8kB 0 1 2 3 4 5 ... 995 996 997 998 999
│         * log_kernel_param_dim_0  (log_kernel_param_dim_0) int64 16B 0 1
│         * mean_dim_0              (mean_dim_0) int64 16B 0 1
│       Data variables:
│           lag                     (chain, draw) float64 8kB ...
│           log_amp_scale           (chain, draw) float64 8kB ...
│           log_drw_scale           (chain, draw) float64 8kB ...
│           log_drw_sigma           (chain, draw) float64 8kB ...
│           log_kernel_param        (chain, draw, log_kernel_param_dim_0) float64 16kB ...
│           mean                    (chain, draw, mean_dim_0) float64 16kB ...
│       Attributes:
│           created_at:                 2026-03-11T21:18:01.611898+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.0.0
│           creation_library_language:  Python
│           inference_library:          numpyro
│           inference_library_version:  0.19.0
├── Group: /sample_stats
│       Dimensions:    (chain: 1, draw: 1000)
│       Coordinates:
│         * chain      (chain) int64 8B 0
│         * draw       (draw) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999
│       Data variables:
│           diverging  (chain, draw) bool 1kB ...
│       Attributes:
│           created_at:                 2026-03-11T21:18:01.627407+00:00
│           creation_library:           ArviZ
│           creation_library_version:   1.0.0
│           creation_library_language:  Python
│           inference_library:          numpyro
│           inference_library_version:  0.19.0
└── Group: /observed_data
        Dimensions:   (gp_dim_0: 32)
        Coordinates:
          * gp_dim_0  (gp_dim_0) int64 256B 0 1 2 3 4 5 6 7 ... 24 25 26 27 28 29 30 31
        Data variables:
            gp        (gp_dim_0) float32 128B ...
        Attributes:
            created_at:                 2026-03-11T21:18:01.633068+00:00
            creation_library:           ArviZ
            creation_library_version:   1.0.0
            creation_library_language:  Python
            inference_library:          numpyro
            inference_library_version:  0.19.0